# BhramGuard URL Model Training

Train a phishing classifier for URLs from `backend/ml/datasets/urls.csv`.

This notebook starts with a URL baseline: character-level TF-IDF plus lexical/structural URL features. It saves model artifacts into `backend/ml/models`.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

DATASET_PATH = PROJECT_ROOT / "backend" / "ml" / "datasets" / "urls.csv"
MODEL_DIR = PROJECT_ROOT / "backend" / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("Models:", MODEL_DIR)

In [ ]:
import ipaddress
import re
from urllib.parse import urlparse

import joblib
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

In [ ]:
df_url = pd.read_csv(DATASET_PATH)

print("Shape:", df_url.shape)
print("Columns:", df_url.columns.tolist())
df_url.head()

In [ ]:
def infer_columns(df):
    label_candidates = ["label", "Label", "class", "target", "is_phishing"]
    url_candidates = ["url", "URL", "text", "link", "domain"]

    label_col = next((col for col in label_candidates if col in df.columns), None)
    if label_col is None:
        raise ValueError(f"No label column found. Columns: {df.columns.tolist()}")

    url_col = next((col for col in url_candidates if col in df.columns and col != label_col), None)
    if url_col is None:
        object_cols = [col for col in df.columns if col != label_col and df[col].dtype == "object"]
        if not object_cols:
            raise ValueError(f"No URL column found. Columns: {df.columns.tolist()}")
        url_col = object_cols[0]

    return url_col, label_col

URL_COL, LABEL_COL = infer_columns(df_url)
print("URL column:", URL_COL)
print("Label column:", LABEL_COL)
print(df_url[LABEL_COL].value_counts(dropna=False))

In [ ]:
df_url = df_url[[URL_COL, LABEL_COL]].dropna()
df_url[URL_COL] = df_url[URL_COL].astype(str).str.strip()
df_url[LABEL_COL] = df_url[LABEL_COL].astype(int)
df_url = df_url.drop_duplicates(subset=[URL_COL])

print("Cleaned shape:", df_url.shape)
print(df_url[LABEL_COL].value_counts(normalize=True))

In [ ]:
def parse_url(value):
    url = "" if pd.isna(value) else str(value).strip()
    parsed = urlparse(url if re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", url) else "http://" + url)
    return url, parsed

def hostname_is_ip(hostname):
    if not hostname:
        return 0
    try:
        ipaddress.ip_address(hostname)
        return 1
    except ValueError:
        return 0

def extract_url_features(value):
    url, parsed = parse_url(value)
    lowered = url.lower()
    hostname = parsed.hostname or ""
    path = parsed.path or ""
    query = parsed.query or ""
    suspicious_words = ["login", "verify", "secure", "account", "update", "confirm", "bank", "wallet", "password"]

    return {
        "url_length": len(url),
        "hostname_length": len(hostname),
        "path_length": len(path),
        "query_length": len(query),
        "dot_count": url.count("."),
        "hyphen_count": url.count("-"),
        "slash_count": url.count("/"),
        "question_count": url.count("?"),
        "equals_count": url.count("="),
        "at_count": url.count("@"),
        "percent_count": url.count("%"),
        "digit_count": sum(char.isdigit() for char in url),
        "has_https": int(parsed.scheme == "https"),
        "has_ip_hostname": hostname_is_ip(hostname),
        "subdomain_count": max(hostname.count(".") - 1, 0),
        "suspicious_word_count": sum(1 for word in suspicious_words if word in lowered),
    }

manual_features = pd.DataFrame(df_url[URL_COL].apply(extract_url_features).tolist())
manual_features.describe().T

In [ ]:
X_url_train, X_url_test, X_manual_train, X_manual_test, y_train, y_test = train_test_split(
    df_url[URL_COL],
    manual_features,
    df_url[LABEL_COL],
    test_size=0.2,
    random_state=42,
    stratify=df_url[LABEL_COL],
)

tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=5000,
    min_df=2,
    lowercase=True,
)

X_train_tfidf = tfidf.fit_transform(X_url_train)
X_test_tfidf = tfidf.transform(X_url_test)

X_train = hstack([X_train_tfidf, csr_matrix(X_manual_train.values)])
X_test = hstack([X_test_tfidf, csr_matrix(X_manual_test.values)])

print("Train matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

In [ ]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "linear_svc": LinearSVC(class_weight="balanced", random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1),
}

results = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    auc = roc_auc_score(y_test, probabilities) if probabilities is not None else None

    print("\n===", model_name, "===")
    print(classification_report(y_test, predictions))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))
    if auc is not None:
        print("ROC-AUC:", auc)

    report = classification_report(y_test, predictions, output_dict=True)
    results[model_name] = {"model": model, "auc": auc, "f1": report["weighted avg"]["f1-score"]}

best_model_name = max(results, key=lambda name: (results[name]["auc"] or 0, results[name]["f1"]))
best_model = results[best_model_name]["model"]
print("Best model:", best_model_name)

In [ ]:
artifact = {
    "model": best_model,
    "vectorizer": tfidf,
    "manual_feature_columns": manual_features.columns.tolist(),
    "url_column": URL_COL,
    "label_column": LABEL_COL,
    "best_model_name": best_model_name,
}

joblib.dump(artifact, MODEL_DIR / "url_phishing_model.pkl")
print("Saved:", MODEL_DIR / "url_phishing_model.pkl")